# WELCOME TO CRESENCIO'S ADDITIONAL EXERCISES FOR LAB 10: DATABASES AND SQL WITH PYTHON

**Additional Exercise 1: Course Enrollment Tracker**
Functionality: Build a CLI program that allows you to register students into various courses using a many-tomany relationship.

Input:

• Student details: full name

• Course list: predefined

• Course enrollment by selecting course IDs

Calculations:

• Use a junction table (enrollments)

• Check for duplicate entries

• Use parameterized queries to protect against SQL injection

Data Size:

• Students table: minimum 10 students

• Courses table: at least 5 courses

• Enrollments: minimum 15 entries (multiple students enrolled in multiple courses)

Output:

• List all courses a student is enrolled in

• List all students enrolled in a specific course

• Display warning for duplicate enrollment attempts

In [6]:
# Exercise 1: Course Enrollment Tracker

import sqlite3
from datetime import datetime

# Database setup
def init_db():
    conn = sqlite3.connect("enrollment.db")
    cursor = conn.cursor()

    # Students table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            full_name TEXT NOT NULL,
            student_id TEXT UNIQUE NOT NULL
        )
    """)

    # Courses table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS courses (
            course_id INTEGER PRIMARY KEY AUTOINCREMENT,
            course_code TEXT UNIQUE NOT NULL,
            description TEXT NOT NULL,
            units INTEGER NOT NULL
        )
    """)

    # Junction table for many-to-many relationship
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS enrollments (
            enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            student_id INTEGER NOT NULL,
            course_id INTEGER NOT NULL,
            enroll_date TEXT NOT NULL,
            FOREIGN KEY (student_id) REFERENCES students(id),
            FOREIGN KEY (course_id) REFERENCES courses(course_id),
            UNIQUE(student_id, course_id)  -- Prevent duplicate enrollments
        )
    """)

    conn.commit()
    return conn, cursor

def populate_sample_data(cursor, conn):
    """Populate with minimum required data"""

    # Check if already populated
    cursor.execute("SELECT COUNT(*) FROM students")
    if cursor.fetchone()[0] > 0:
        return

    # Insert 12 students (K-pop idols edition)
    students = [
        ("Son Chaeyoung", "2021-00001"),
        ("Ning Yizhuo", "2021-00002"),
        ("Yu Ji-min", "2021-00003"),
        ("Enami Asa", "2021-00004"),
        ("Aeri Uchinaga", "2021-00005"),
        ("Kim Min-jeong", "2021-00006"),
        ("Kim Chaewon", "2021-00007"),
        ("Hwang Ye-ji", "2021-00008"),
        ("Rei Naoi", "2021-00009"),
        ("Bae Joo-hyun", "2021-00010"),
        ("Chou Tzu-yu", "2021-00011"),
        ("Sana Minatozaki", "2021-00012")
    ]
    cursor.executemany("INSERT OR IGNORE INTO students (full_name, student_id) VALUES (?, ?)", students)

    # Insert 7 courses (FIXED: removed duplicate ECE101)
    courses = [
        ("ECE101", "Electronics Principles", 3),
        ("ECE201", "Digital Signal Processing", 3),
        ("EE101", "Electrical Circuits", 3),
        ("COE101", "Computer Organization", 3),
        ("ECE301", "Signals and Systems", 3),      # FIXED: was duplicate ECE101
        ("MATH201", "Advanced Engineering Math", 4),
        ("PHYS101", "Physics for Engineers", 3)
    ]
    cursor.executemany("INSERT OR IGNORE INTO courses (course_code, description, units) VALUES (?, ?, ?)", courses)

    # Insert 20 enrollments (FIXED comments to match actual students)
    enrollments = [
        (1, 1, datetime.now().strftime("%Y-%m-%d")),   # Chaeyoung -> Electronics
        (1, 2, datetime.now().strftime("%Y-%m-%d")),   # Chaeyoung -> DSP
        (2, 1, datetime.now().strftime("%Y-%m-%d")),   # Ningning -> Electronics
        (2, 3, datetime.now().strftime("%Y-%m-%d")),   # Ningning -> Electrical
        (3, 4, datetime.now().strftime("%Y-%m-%d")),   # Karina -> Comp Org
        (3, 5, datetime.now().strftime("%Y-%m-%d")),   # Karina -> Signals
        (4, 1, datetime.now().strftime("%Y-%m-%d")),   # Asa -> Electronics
        (4, 6, datetime.now().strftime("%Y-%m-%d")),   # Asa -> Math
        (5, 2, datetime.now().strftime("%Y-%m-%d")),   # Giselle -> DSP
        (5, 7, datetime.now().strftime("%Y-%m-%d")),   # Giselle -> Physics
        (6, 3, datetime.now().strftime("%Y-%m-%d")),   # Winter -> Electrical
        (6, 4, datetime.now().strftime("%Y-%m-%d")),   # Winter -> Comp Org
        (7, 1, datetime.now().strftime("%Y-%m-%d")),   # Chaewon -> Electronics
        (7, 6, datetime.now().strftime("%Y-%m-%d")),   # Chaewon -> Math
        (8, 2, datetime.now().strftime("%Y-%m-%d")),   # Yeji -> DSP
        (8, 5, datetime.now().strftime("%Y-%m-%d")),   # Yeji -> Signals
        (9, 3, datetime.now().strftime("%Y-%m-%d")),   # Rei -> Electrical
        (9, 7, datetime.now().strftime("%Y-%m-%d")),   # Rei -> Physics
        (10, 4, datetime.now().strftime("%Y-%m-%d")),  # Irene -> Comp Org
        (10, 1, datetime.now().strftime("%Y-%m-%d"))   # Irene -> Electronics
    ]
    cursor.executemany("""
        INSERT OR IGNORE INTO enrollments (student_id, course_id, enroll_date) 
        VALUES (?, ?, ?)
    """, enrollments)

    conn.commit()
    print("✓ Sample data populated successfully!")

def list_student_courses(cursor, student_name):
    """List all courses a specific student is enrolled in"""
    cursor.execute("""
        SELECT s.full_name, c.course_code, c.description, c.units, e.enroll_date
        FROM students s
        JOIN enrollments e ON s.id = e.student_id
        JOIN courses c ON e.course_id = c.course_id
        WHERE s.full_name = ?
        ORDER BY c.course_code
    """, (student_name,))

    results = cursor.fetchall()
    if not results:
        print(f"\n No courses found for student: {student_name}")
        return

    print(f"\n Courses enrolled by {student_name}:")
    print("-" * 70)
    print(f"{'Course Code':<12} {'Description':<30} {'Units':<6} {'Date Enrolled'}")
    print("-" * 70)
    total_units = 0
    for row in results:
        print(f"{row[1]:<12} {row[2]:<30} {row[3]:<6} {row[4]}")
        total_units += row[3]
    print("-" * 70)
    print(f"Total Units: {total_units}")

def list_course_students(cursor, course_code):
    """List all students enrolled in a specific course"""
    cursor.execute("""
        SELECT c.course_code, c.description, s.full_name, s.student_id, e.enroll_date
        FROM courses c
        JOIN enrollments e ON c.course_id = e.course_id
        JOIN students s ON e.student_id = s.id
        WHERE c.course_code = ?
        ORDER BY s.full_name
    """, (course_code,))

    results = cursor.fetchall()
    if not results:
        print(f"\n No students found for course: {course_code}")
        return

    print(f"\n Students enrolled in {course_code} - {results[0][1]}:")
    print("-" * 60)
    print(f"{'Student Name':<25} {'Student ID':<15} {'Date Enrolled'}")
    print("-" * 60)
    for row in results:
        print(f"{row[2]:<25} {row[3]:<15} {row[4]}")
    print(f"\nTotal students: {len(results)}")

def enroll_student(cursor, conn, student_name, course_code):
    """Enroll a student in a course with duplicate checking"""
    try:
        # Get student ID
        cursor.execute("SELECT id FROM students WHERE full_name = ?", (student_name,))
        student = cursor.fetchone()
        if not student:
            print(f" Error: Student '{student_name}' not found!")
            return False

        # Get course ID
        cursor.execute("SELECT course_id FROM courses WHERE course_code = ?", (course_code,))
        course = cursor.fetchone()
        if not course:
            print(f" Error: Course '{course_code}' not found!")
            return False

        student_id, course_id = student[0], course[0]

        # Attempt enrollment with parameterized query
        cursor.execute("""
            INSERT INTO enrollments (student_id, course_id, enroll_date)
            VALUES (?, ?, ?)
        """, (student_id, course_id, datetime.now().strftime("%Y-%m-%d")))

        conn.commit()
        print(f" Successfully enrolled {student_name} in {course_code}!")
        return True

    except sqlite3.IntegrityError:
        print(f" Warning: {student_name} is already enrolled in {course_code}!")
        return False

def display_menu():
    print("\n" + "="*50)
    print("   COURSE ENROLLMENT TRACKER SYSTEM")
    print("="*50)
    print("1. List all courses for a student")
    print("2. List all students in a course")
    print("3. Enroll a student in a course")
    print("4. View all students")
    print("5. View all courses")
    print("6. Exit")
    print("="*50)

def main():
    conn, cursor = init_db()
    populate_sample_data(cursor, conn)

    while True:
        display_menu()
        choice = input("Enter choice (1-6): ").strip()

        if choice == "1":
            name = input("Enter student full name: ").strip()
            list_student_courses(cursor, name)

        elif choice == "2":
            code = input("Enter course code (e.g., ECE101): ").strip()
            list_course_students(cursor, code)

        elif choice == "3":
            name = input("Enter student full name: ").strip()
            code = input("Enter course code: ").strip()
            enroll_student(cursor, conn, name, code)

        elif choice == "4":
            cursor.execute("SELECT * FROM students ORDER BY full_name")
            print("\n All Students:")
            for row in cursor.fetchall():
                print(f"  ID: {row[0]}, Name: {row[1]}, Student ID: {row[2]}")

        elif choice == "5":
            cursor.execute("SELECT * FROM courses ORDER BY course_code")
            print("\n All Courses:")
            for row in cursor.fetchall():
                print(f"  Code: {row[1]}, Description: {row[2]}, Units: {row[3]}")

        elif choice == "6":
            print("\nGoodbye! ")
            break
        else:
            print(" Invalid choice. Please try again.")

    conn.close()

if __name__ == "__main__":
    main()


   COURSE ENROLLMENT TRACKER SYSTEM
1. List all courses for a student
2. List all students in a course
3. Enroll a student in a course
4. View all students
5. View all courses
6. Exit



 All Students:
  ID: 5, Name: Aeri Uchinaga, Student ID: 2021-00005
  ID: 10, Name: Bae Joo-hyun, Student ID: 2021-00010
  ID: 11, Name: Chou Tzu-yu, Student ID: 2021-00011
  ID: 4, Name: Enami Asa, Student ID: 2021-00004
  ID: 8, Name: Hwang Ye-ji, Student ID: 2021-00008
  ID: 7, Name: Kim Chaewon, Student ID: 2021-00007
  ID: 6, Name: Kim Min-jeong, Student ID: 2021-00006
  ID: 2, Name: Ning Yizhuo, Student ID: 2021-00002
  ID: 9, Name: Rei Naoi, Student ID: 2021-00009
  ID: 12, Name: Sana Minatozaki, Student ID: 2021-00012
  ID: 1, Name: Son Chaeyoung, Student ID: 2021-00001
  ID: 3, Name: Yu Ji-min, Student ID: 2021-00003

   COURSE ENROLLMENT TRACKER SYSTEM
1. List all courses for a student
2. List all students in a course
3. Enroll a student in a course
4. View all students
5. View all courses
6. Exit
 Error: Student '2' not found!

   COURSE ENROLLMENT TRACKER SYSTEM
1. List all courses for a student
2. List all students in a course
3. Enroll a student in a course
4. View all s

**Additional Exercise 2: Attendance Recording System**

Functionality:

Design a CLI-based attendance tracker that marks and stores each student’s presence or absence by date.

Input:

• Student name or ID

• Date

• Attendance status: Present or Absent

Calculations:

• Store attendance records in a related table

• Filter by student or date

• Use DATE() function for date processing

Output:

• Dashboard view with the uploaded profile picture and updated name

• Flash message on successful profile update

• Image rendered using 
<img src="{{ url_for('static', filename='uploads/' + user.image_filename) }}">

In [8]:
# Exercise 2: Attendance Recording System
import sqlite3
from datetime import datetime, date

def init_db():
    conn = sqlite3.connect("attendance.db")
    cursor = conn.cursor()

    # Students table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            full_name TEXT NOT NULL,
            student_id TEXT UNIQUE NOT NULL
        )
    """)

    # Attendance records table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS attendance (
            attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
            student_id INTEGER NOT NULL,
            attendance_date DATE NOT NULL,
            status TEXT NOT NULL CHECK(status IN ('Present', 'Absent')),
            recorded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (student_id) REFERENCES students(id),
            UNIQUE(student_id, attendance_date)  -- One record per student per day
        )
    """)

    conn.commit()
    return conn, cursor

def populate_sample_students(cursor, conn):
    """Add sample students if table is empty"""
    cursor.execute("SELECT COUNT(*) FROM students")
    if cursor.fetchone()[0] > 0:
        return

    # K-pop idols edition
    students = [
        ("Son Chaeyoung", "2021-00001"),
        ("Ning Yizhuo", "2021-00002"),
        ("Yu Ji-min", "2021-00003"),
        ("Enami Asa", "2021-00004"),
        ("Aeri Uchinaga", "2021-00005"),
        ("Kim Min-jeong", "2021-00006"),
        ("Kim Chaewon", "2021-00007"),
        ("Hwang Ye-ji", "2021-00008"),
        ("Rei Naoi", "2021-00009"),
        ("Bae Joo-hyun", "2021-00010")
    ]
    cursor.executemany("INSERT INTO students (full_name, student_id) VALUES (?, ?)", students)
    conn.commit()
    print("✓ Sample students added!")

def mark_attendance(cursor, conn, student_name, att_date, status):
    """Mark attendance for a student on a specific date"""
    try:
        # Get student ID
        cursor.execute("SELECT id FROM students WHERE full_name = ?", (student_name,))
        student = cursor.fetchone()
        if not student:
            print(f"⚠ Student '{student_name}' not found!")
            return False

        student_id = student[0]

        # Use parameterized query to prevent SQL injection
        cursor.execute("""
            INSERT INTO attendance (student_id, attendance_date, status)
            VALUES (?, ?, ?)
        """, (student_id, att_date, status))

        conn.commit()
        print(f"✓ Attendance marked: {student_name} - {status} on {att_date}")
        return True

    except sqlite3.IntegrityError:
        print(f"⚠ Attendance already recorded for {student_name} on {att_date}!")
        return False

def get_attendance_by_student(cursor, student_name):
    """Get all attendance records for a specific student"""
    cursor.execute("""
        SELECT s.full_name, a.attendance_date, a.status, a.recorded_at
        FROM students s
        JOIN attendance a ON s.id = a.student_id
        WHERE s.full_name = ?
        ORDER BY a.attendance_date DESC
    """, (student_name,))

    results = cursor.fetchall()
    if not results:
        print(f"\n⚠ No attendance records found for {student_name}")
        return

    print(f"\n📊 Attendance Record for {student_name}:")
    print("-" * 60)
    print(f"{'Date':<15} {'Status':<10} {'Recorded At'}")
    print("-" * 60)

    present = 0
    absent = 0
    for row in results:
        print(f"{row[1]:<15} {row[2]:<10} {row[3]}")
        if row[2] == 'Present':
            present += 1
        else:
            absent += 1

    print("-" * 60)
    total = present + absent
    if total > 0:
        rate = (present / total) * 100
        print(f"Summary: Present: {present}, Absent: {absent}")
        print(f"Attendance Rate: {rate:.1f}%")

def get_attendance_by_date(cursor, att_date):
    """Get all attendance records for a specific date"""
    cursor.execute("""
        SELECT s.full_name, a.status
        FROM students s
        JOIN attendance a ON s.id = a.student_id
        WHERE a.attendance_date = ?
        ORDER BY s.full_name
    """, (att_date,))

    results = cursor.fetchall()
    if not results:
        print(f"\n⚠ No attendance records found for {att_date}")
        return

    print(f"\n📅 Attendance for {att_date}:")
    print("-" * 40)
    print(f"{'Student Name':<25} {'Status'}")
    print("-" * 40)

    present = 0
    for row in results:
        print(f"{row[0]:<25} {row[1]}")
        if row[1] == 'Present':
            present += 1

    print("-" * 40)
    print(f"Total Present: {present}/{len(results)}")

def get_monthly_summary(cursor, month, year):
    """Get attendance summary for a specific month using DATE() function"""
    cursor.execute("""
        SELECT 
            s.full_name,
            COUNT(CASE WHEN a.status = 'Present' THEN 1 END) as present_count,
            COUNT(CASE WHEN a.status = 'Absent' THEN 1 END) as absent_count,
            COUNT(*) as total_days
        FROM students s
        LEFT JOIN attendance a ON s.id = a.student_id 
            AND strftime('%m', a.attendance_date) = ?
            AND strftime('%Y', a.attendance_date) = ?
        GROUP BY s.id, s.full_name
        ORDER BY s.full_name
    """, (month, year))

    results = cursor.fetchall()
    print(f"\n📈 Monthly Attendance Summary ({month}/{year}):")
    print("-" * 70)
    print(f"{'Student Name':<25} {'Present':<8} {'Absent':<8} {'Total':<8} {'Rate'}")
    print("-" * 70)

    for row in results:
        name, present, absent, total = row
        rate = (present / total * 100) if total > 0 else 0
        print(f"{name:<25} {present:<8} {absent:<8} {total:<8} {rate:.1f}%")

def display_menu():
    print("\n" + "="*50)
    print("   ATTENDANCE RECORDING SYSTEM")
    print("="*50)
    print("1. Mark attendance (Present/Absent)")
    print("2. View attendance by student")
    print("3. View attendance by date")
    print("4. Monthly summary report")
    print("5. View all students")
    print("6. Bulk mark attendance for today")
    print("7. Exit")
    print("="*50)

def bulk_mark_today(cursor, conn):
    """Quickly mark all students for today"""
    today = date.today().isoformat()
    print(f"\n📅 Bulk attendance for {today}:")
    print("Enter 'p' for Present, 'a' for Absent, or press Enter to skip")

    cursor.execute("SELECT id, full_name FROM students ORDER BY full_name")
    students = cursor.fetchall()

    for student_id, name in students:
        status_input = input(f"  {name}: ").strip().lower()
        if status_input == 'p':
            mark_attendance(cursor, conn, name, today, 'Present')
        elif status_input == 'a':
            mark_attendance(cursor, conn, name, today, 'Absent')
        else:
            print(f"    Skipped {name}")

def main():
    conn, cursor = init_db()
    populate_sample_students(cursor, conn)

    # Add some sample attendance data for demonstration
    sample_dates = ['2026-04-28', '2026-04-29', '2026-04-30']
    for i, sample_date in enumerate(sample_dates):
        try:
            cursor.execute("""
                INSERT OR IGNORE INTO attendance (student_id, attendance_date, status)
                VALUES (?, ?, ?)
            """, (1, sample_date, 'Present' if i % 2 == 0 else 'Absent'))
            cursor.execute("""
                INSERT OR IGNORE INTO attendance (student_id, attendance_date, status)
                VALUES (?, ?, ?)
            """, (2, sample_date, 'Present'))
        except:
            pass
    conn.commit()

    while True:
        display_menu()
        choice = input("Enter choice (1-7): ").strip()

        if choice == "1":
            name = input("Enter student full name: ").strip()
            att_date = input("Enter date (YYYY-MM-DD) or press Enter for today: ").strip()
            if not att_date:
                att_date = date.today().isoformat()
            status = input("Status (Present/Absent): ").strip().capitalize()
            if status in ['Present', 'Absent']:
                mark_attendance(cursor, conn, name, att_date, status)
            else:
                print("⚠ Invalid status! Use 'Present' or 'Absent'")

        elif choice == "2":
            name = input("Enter student full name: ").strip()
            get_attendance_by_student(cursor, name)

        elif choice == "3":
            att_date = input("Enter date (YYYY-MM-DD): ").strip()
            get_attendance_by_date(cursor, att_date)

        elif choice == "4":
            month = input("Enter month (01-12): ").strip()
            year = input("Enter year (YYYY): ").strip()
            get_monthly_summary(cursor, month, year)

        elif choice == "5":
            cursor.execute("SELECT * FROM students ORDER BY full_name")
            print("\n📋 All Students:")
            for row in cursor.fetchall():
                print(f"  ID: {row[0]}, Name: {row[1]}, Student ID: {row[2]}")

        elif choice == "6":
            bulk_mark_today(cursor, conn)

        elif choice == "7":
            print("\nGoodbye! 👋")
            break
        else:
            print("⚠ Invalid choice. Please try again.")

    conn.close()

if __name__ == "__main__":
    main()


   ATTENDANCE RECORDING SYSTEM
1. Mark attendance (Present/Absent)
2. View attendance by student
3. View attendance by date
4. Monthly summary report
5. View all students
6. Bulk mark attendance for today
7. Exit

📊 Attendance Record for Ning Yizhuo:
------------------------------------------------------------
Date            Status     Recorded At
------------------------------------------------------------
2026-04-30      Present    2026-05-01 17:29:26
2026-04-29      Present    2026-05-01 17:29:26
2026-04-28      Present    2026-05-01 17:29:26
------------------------------------------------------------
Summary: Present: 3, Absent: 0
Attendance Rate: 100.0%

   ATTENDANCE RECORDING SYSTEM
1. Mark attendance (Present/Absent)
2. View attendance by student
3. View attendance by date
4. Monthly summary report
5. View all students
6. Bulk mark attendance for today
7. Exit

Goodbye! 👋


**Additional Exercise 3: Profile Management with Image Upload**

Functionality: Build a profile management feature where users can update their display name and upload a profile picture. Uploaded images are stored in the /static/uploads directory and displayed on their dashboard.
 
Input:

• Display name input field (text)

• File upload input (only .jpg, .png, .jpeg formats allowed)

Calculations:

• Check file extension and MIME type for validation

• Generate a secure filename using werkzeug.utils.secure_filename

• Save the image to a static uploads directory


• Store image filename and display name in the user profile table

• Ensure users can only update their profiles using current_user.id

Output:

• Dashboard view with the uploaded profile picture and updated name

• Flash message on successful profile update

• Image rendered using
<img src="{{ url_for('static', filename='uploads/' + user.image_filename) }}">

**FOR THE SOLUTION AND OUTPUT OF ADDITIONAL EXERCISE NUMBER 3 PROCEED TO "Additional_Exercise3.py"**

# THE END OF CRESENCIO'S LAB 10 ADDITIONAL EXERCISE.